# Project 5 NLP — Pass 01

## Problem Definition


State the business problem. Translate the business problem into a Data Science problem by stating what kind of problem it is ( supervised vs unsupervised ) and whether it is a classification, regression, or clustering problem.

## Data Collection/Sources


In [35]:
%%capture output
#install Wikipedia API
!pip3 install wikipedia-api

In [86]:
import numpy as np
import pandas as pd
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors

pd.options.display.max_columns = 100

import nltk
nltk.download('punkt_tab')
import wikipediaapi

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### An initial peek at the data


In [39]:
# define url for data source
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv'

In [42]:
! curl -s {url} | wc -l


42786


In [43]:
%%bash -s "$url"
curl -s $1 |
head -1 |
tr , '\n' |
cat -n

     1	URI
     2	name
     3	text


In [44]:
%%bash -s "$url"
curl -s $1 |
sed -ne '14p' |
tr , '\n' |
cat -n

     1	<http://dbpedia.org/resource/Jonathan_Hoefler>
     2	Jonathan Hoefler
     3	jonathan hoefler born august 22 1970 is an american typeface designer hoefler pronounced heffler founded the hoefler type foundry in 1989 a type foundry in new york in 1999 hoefler began working with type designer tobias frerejones and from 20052014 the company operated under the name hoefler frerejones until their public splithoefler has designed original typefaces for rolling stone magazine harpers bazaar the new york times magazine sports illustrated and esquire and several institutional clients including the solomon r guggenheim museum and alternative band they might be giants perhaps his bestknown work is the hoefler text family of typefaces designed for apple computer and now appearing as part of the macintosh operating system he also designed the current wordmark of the church of the latterday saintsin 1995 hoefler was named one of the forty most influential designers in america by id magazine a

In [45]:
%%bash -s "$url"
curl -s $1 |
sed -ne '1p; 100p;' |
cut -c1-120 |
cat -n

     1	URI,name,text
     2	<http://dbpedia.org/resource/John_Reynolds_(Canadian_politician)>,John Reynolds (Canadian politician),john douglas reyno


In [98]:
%%bash -s "$url"
curl -s $1 |
head -21 |
cut -c1-120 |
cat -n

     1	URI,name,text
     2	<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former australian rule
     3	<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from university of ch
     4	<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player who has been ac
     5	<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lower au
     6	<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn better known by his stagename 
     7	<http://dbpedia.org/resource/Sam_Henderson>,Sam Henderson,sam henderson born october 18 1969 is an american cartoonist w
     8	<http://dbpedia.org/resource/Aaron_LaCrate>,Aaron LaCrate,aaron lacrate is an american music producer recording artist d
     9	<http://dbpedia.org/resource/Trevor_Ferguson>,Trevor Ferguson,trevor ferguso

### Retrieving and Importing the Data

In [99]:
# set number of rows to import from csv file
num_rows_to_import = 20

In [100]:
# load rows into pandas dataframe
df = pd.read_csv(url, nrows = num_rows_to_import)

### Data Inspection

In [101]:
df.head()

,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...


In [102]:
df.tail()

,URI,name,text
15,<http://dbpedia.org/resource/Joerg_Steineck>,Joerg Steineck,joerg steineck is a german filmmaker editor an...
16,<http://dbpedia.org/resource/Andrew_Pinsent>,Andrew Pinsent,fr andrew pinsent born 19 august 1966 is resea...
17,<http://dbpedia.org/resource/Paddy_Dunne_(Gael...,Paddy Dunne (Gaelic footballer),paddy dunne was a gaelic football player from ...
18,<http://dbpedia.org/resource/Alexandros_Mouzas>,Alexandros Mouzas,alexandros mouzas born 1962 is a greek compose...
19,<http://dbpedia.org/resource/John_Angus_Campbell>,John Angus Campbell,john angus campbell born march 10 1942 in port...


In [103]:
df

,URI,name,text
0,<http://dbpedia.org/resource/Digby_Morrell>,Digby Morrell,digby morrell born 10 october 1979 is a former...
1,<http://dbpedia.org/resource/Alfred_J._Lewy>,Alfred J. Lewy,alfred j lewy aka sandy lewy graduated from un...
2,<http://dbpedia.org/resource/Harpdog_Brown>,Harpdog Brown,harpdog brown is a singer and harmonica player...
3,<http://dbpedia.org/resource/Franz_Rottensteiner>,Franz Rottensteiner,franz rottensteiner born in waidmannsfeld lowe...
4,<http://dbpedia.org/resource/G-Enka>,G-Enka,henry krvits born 30 december 1974 in tallinn ...
5,<http://dbpedia.org/resource/Sam_Henderson>,Sam Henderson,sam henderson born october 18 1969 is an ameri...
6,<http://dbpedia.org/resource/Aaron_LaCrate>,Aaron LaCrate,aaron lacrate is an american music producer re...
7,<http://dbpedia.org/resource/Trevor_Ferguson>,Trevor Ferguson,trevor ferguson aka john farrow born 11 novemb...
8,<http://dbpedia.org/resource/Grant_Nelson>,Grant Nelson,grant nelson born 27 april 1971 in london also...
9,<http://dbpedia.org/resource/Cathy_Caruth>,Cathy Caruth,cathy caruth born 1955 is frank h t rhodes pro...


In [19]:
df.size

60

In [20]:
df.shape

(20, 3)

In [28]:
# info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   URI     20 non-null     object
 1   name    20 non-null     object
 2   text    20 non-null     object
dtypes: object(3)
memory usage: 612.0+ bytes


In [29]:
# describe
df.describe().transpose()

,count,unique,top,freq
URI,20,20,<http://dbpedia.org/resource/Digby_Morrell>,1
name,20,20,Digby Morrell,1
text,20,20,digby morrell born 10 october 1979 is a former...,1


In [30]:
# look for nulls
df.isnull().sum().sort_values(ascending=False)

,0
URI,0
name,0
text,0


In [31]:
df.nunique()

,0
URI,20
name,20
text,20


## Data Cleaning


#### Preprocessing

In [87]:
# No preprocessing required for this pass.

#### BoW ( Bag of Words Using CountVectorizer)

In [33]:
# Pick person to use as reference for analysis
reference_person = "Jonathan Hoefler"

In [76]:
# look at indices for df to see how we can access the row at reference_person
df.index

RangeIndex(start=0, stop=20, step=1)

In [49]:
# get the index for row where name is our reference person
reference_index = int(df.index[df['name'] == reference_person][0])
reference_index

np.int64(12)

In [78]:
vectorizer = CountVectorizer(stop_words = 'english')
texts = list(df['text'])
bow_matrix = vectorizer.fit_transform(texts)

In [79]:
type(bow_matrix), bow_matrix.shape

(scipy.sparse._csr.csr_matrix, (20, 2004))

In [80]:
bow_matrix.toarray()

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0]], shape=(20, 2004))

#### TF-IDF using BoW

In [81]:
tf_idf_matrix = TfidfTransformer()
tf_idf_texts = tf_idf_matrix.fit_transform(bow_matrix)

In [82]:
type(tf_idf_texts), tf_idf_texts.shape

(scipy.sparse._csr.csr_matrix, (20, 2004))

In [83]:
tf_idf_texts.toarray()

array([[0.04455466, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.05585618, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(20, 2004))

In [84]:
tf_df = pd.DataFrame(
  data = tf_idf_texts.toarray(),
  columns = vectorizer.get_feature_names_out(),
)

In [85]:
tf_df

,10,1000,102,11,12,1406,17,173182,17th,18,18th,19,1939,1942,1947,1949,1950swith,1951,1952,1953,1953called,1954,1955,1959,1962,1963,1966,19681995,1969,1970,1971,1973,1974,1975,1977,1978,1979,19791985,1980,1980s,1981,1982,1985,1987,1988,1989,19901993,1990s,1991,1991henderson,...,widely,wimpey,winner,winning,winter,winters,wire,wishdokta,withdrew,witness,woman,won,wont,wood,woodland,woodstock,wordless,wordmark,wordsworth,work,worked,worker,working,works,world,worlds,write,writer,writerinresidence,writers,writersextraordinary,writing,written,wrote,wutang,xanthoudakis,xeroxed,xv,yale,year,years,york,yorks,yorksince,young,youngest,zarathustra,zelazny,ziet,zwigoff
0,0.044555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.06132,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.060341,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.060341,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.068646,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.042986,0.042986,0.000000,0.042986,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.064978,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049689,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.046289,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.053711,0.046289,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.046945,0.000000,0.000000,0.000000,0.052025,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.052025,0.000000,0.000000,0.052025,0.00000,0.000000,0.052025,0.000000,0.118372,0.000000,0.059186,0.00000,0.059186,0.052025,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.037

## Exploratory Data Analysis


In [ ]:
# No EDA required for this pass.

## Processing



#### K Nearest Neighbors

In [94]:
num_neighbors = 10

In [91]:
nn = NearestNeighbors().fit(tf_idf_texts)

In [92]:
# create the reference matrix from the tf_idf matrix
reference_matrix = tf_idf_texts[reference_index]
reference_matrix.shape

(1, 2004)

In [93]:
# alt: create the reference matrix from the tf data frame

# reference_matrix = np.array([tf_df.iloc[reference_index]])
# reference_matrix.shape

In [109]:
# get nearest neighbors distances
distances, indices = nn.kneighbors(
  X = reference_matrix,
  n_neighbors = 1 + num_neighbors
)

In [110]:
# verify
distances, indices

(array([[0.        , 1.3663033 , 1.38506224, 1.38543067, 1.38724507,
         1.39222322, 1.394952  , 1.39761893, 1.39907879, 1.40114063,
         1.40258849]]),
 array([[12,  5,  7, 19, 11,  3,  2,  6, 14, 15,  1]]))

In [111]:
count = 0
np.array([ f"{i}. {str(text)[:50]}..." for i, text in enumerate(texts) ] )[indices]

array([['12. jonathan hoefler born august 22 1970 is an america...',
        '5. sam henderson born october 18 1969 is an american ...',
        '7. trevor ferguson aka john farrow born 11 november 1...',
        '19. john angus campbell born march 10 1942 in portland...',
        '11. jenn ashworth is an english writer she was born in...',
        '3. franz rottensteiner born in waidmannsfeld lower au...',
        '2. harpdog brown is a singer and harmonica player who...',
        '6. aaron lacrate is an american music producer record...',
        '14. david chernushenko born june 1963 in calgary alber...',
        '15. joerg steineck is a german filmmaker editor and gr...',
        '1. alfred j lewy aka sandy lewy graduated from univer...']],
      dtype='<U57')

In [113]:
indices[0][1:]

array([ 5,  7, 19, 11,  3,  2,  6, 14, 15,  1])

In [115]:
[df['name'][x] for x in indices[0][1:]]

['Sam Henderson',
 'Trevor Ferguson',
 'John Angus Campbell',
 'Jenn Ashworth',
 'Franz Rottensteiner',
 'Harpdog Brown',
 'Aaron LaCrate',
 'David Chernushenko',
 'Joerg Steineck',
 'Alfred J. Lewy']

## Data Visualization/Communication of Results
